# FLUX.2 klein 4B · Custom prompts · BF16 / INT8 / INT4 (built-in reference photo · fast INT4)

**The original photo is the reference image, plus your own complete text prompt; 4-step generation of a still image.**
Edit `PROMPT` in section 2, in Chinese or English; add sentences to `EXTRA_PROMPTS` for batch tests.
Every prompt goes to the model verbatim: no translation, no parsing of "fat/strong" keywords, no automatically added waist-up framing, clothing, background, gender or identity-preservation text.
If you want something preserved, say so in the prompt yourself. A dance instruction produces a still image of a dance pose, not a video.

| Group | Image transformer | Qwen text encoder | VAE / unquantized layers / activations |
|---|---|---|---|
| BF16 | BF16 | BF16 | BF16 |
| INT8 | Linear weights qint8 | Linear weights qint8 | BF16 |
| INT4 | Linear weights qint4 (4-bit packed, dequantized before compute) | Linear weights qint4 (same) | BF16 |

**Fast INT4:** the previous INT4 version ran on Quanto's tinygemm kernel (`torch._weight_int4pack_mm`), which is designed for single-token decoding and is about 6x slower on DiT matmuls with more than 8,000 tokens each. This version defaults to `INT4_BACKEND='dequant_bf16'`: quantization still calls Quanto's own `MaxOptimizer` and `quantize_affine`, so the int4 values, scales and shifts are bit-identical to Quanto's; the weights stay resident in GPU memory packed at 4 bits (0.53 bytes per parameter, as before); before each layer computes, three elementwise kernels expand it to BF16 and cuBLAS does the matmul. The dequantization reproduces tinygemm's GPU arithmetic `fma(q-8, scale, zero)`, so the weights fed to the matmul are the same as in the previous INT4 version; only the matmul kernel changed. To switch back to the old path: `INT4_BACKEND='quanto_tinygemm'`.

All three groups are compared by default. To quickly see the effect of a new prompt, set `PRECISION_MODE` to `bf16`.
The original Qwen text encoder is still used; this version does no 4B→0.6B alignment or replacement.
INT4 is Quanto integer weight quantization, not NF4/FP4/FP8. The DiT's `proj_out`, the text encoder's `lm_head`,
embeddings, normalization and convolutions are not quantized; every layer is checked after `freeze()`, with no silent fallback to BF16.

**The reference photo is embedded in this notebook as base64 (original JPEG bytes, not re-encoded); it is decoded and its SHA-256 verified at run time, so no manual upload is needed.** To test extra images as well, turn on `ALSO_UPLOAD` in section 2.

**Run:** pick a Colab A100 runtime (40 GB or more recommended), fill in the prompt, then run from top to bottom.
Every prompt is applied to every image; all three groups share the inputs, seed, resolution and 4-step setting. The model is downloaded once, and each group runs in its own process.
To try new prompts: edit section 2 and re-run that section and everything after it; section 1 (dependencies) does not need to be re-run.

**Inference mode:** all groups use `torch.no_grad()` (the old tinygemm path fails under `inference_mode` with `Cannot set version_counter for inference tensor`). The new INT4 path was validated numerically on CPU with the same torch 2.11 / diffusers 0.39 / transformers 4.57 / Quanto 0.2.7 as Colab; the build environment has no GPU, so speed is whatever this run shows.

Outputs: images, PNG comparison grids, the full actual prompts, time/memory/tensor-storage numbers, HTML, CSV, logs and a ZIP.
BF16 is still loaded before conversion, so the load+quantize peak can be high. This experiment is not a phone deployment package.

## 1. Install dependencies and check the GPU
The PyTorch/CUDA versions already installed in the runtime are kept, not force-upgraded. The low-level Quanto API is used,
not Diffusers' deprecated Quanto loading integration. The default INT4 path does not compile any CUDA extension; only after switching `INT4_BACKEND` back to `quanto_tinygemm` does the first run compile Quanto's extension (about 3–4 minutes on an A100, not counted in the generation time).

In [ ]:
import sys, os, subprocess, importlib.metadata as metadata
from pathlib import Path
from packaging.version import Version

ROOT = Path('/content') if Path('/content').is_dir() else Path.cwd()
try:
    torch_version = metadata.version('torch')
except metadata.PackageNotFoundError:
    raise RuntimeError('PyTorch is not installed in this environment. Select a Colab A100 GPU runtime first.')
if Version(torch_version.split('+')[0]) < Version('2.6'):
    raise RuntimeError('This experiment needs PyTorch >= 2.6; use a newer Colab GPU runtime.')

constraints = ROOT / 'klein_torch_constraints.txt'
keep = []
for name in ('torch', 'torchvision', 'torchaudio'):
    try:
        keep.append(f'{name}=={metadata.version(name)}')
    except metadata.PackageNotFoundError:
        pass
constraints.write_text('\n'.join(keep) + '\n', encoding='utf-8')
requirements = [
    'diffusers>=0.37.1,<1.0', 'transformers>=4.57,<5.0',
    'optimum-quanto==0.2.7', 'accelerate>=1.2,<2', 'safetensors>=0.4',
    'huggingface_hub>=0.34,<1.0', 'ninja', 'Pillow', 'pandas', 'numpy', 'opencv-python-headless',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraints), *requirements])
# Do not create a CUDA context in the notebook process, so it holds no GPU memory while the worker processes run.
check = "import torch\nfrom diffusers import Flux2KleinPipeline\nfrom optimum.quanto import qint4, qint8, freeze, quantize\nassert torch.cuda.is_available(), 'No CUDA GPU. Select an A100 GPU runtime.'\nassert torch.cuda.is_bf16_supported(), 'BF16-capable GPU required (e.g. A100).'\nprint('GPU:', torch.cuda.get_device_name(0))\nprint('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))\nprint('Torch:', torch.__version__, 'CUDA:', torch.version.cuda)\nprint('Flux2KleinPipeline + Quanto imports OK')\n"
subprocess.check_call([sys.executable, '-c', check])
print('Environment ready. The reference photo is built in; fill in the prompt and keep running the cells below.')

## 2. Enter your own prompt (this is the main thing to edit)
Write the complete request in `PROMPT`, replacing the dance example below. In Colab you can use the form field; elsewhere edit the string directly.
Triple-quoted strings work for long multi-line prompts. Each item added to `EXTRA_PROMPTS` tests one more instruction; Chinese and English versions are not generated automatically.

Things to try: change the outfit, change the background, perform an action, change the art style, edit objects, and so on. The results depend on the current klein model.
If you want to keep the face or the clothing, say so in your own text; no hidden template is added here.

`PRECISION_MODE` defaults to `bf16,int8,int4`; choose `bf16` to just test the prompt.
`WIDTH` / `HEIGHT` set the output size (multiples of 16); `REPEATS=3` gives more stable timings.
`QUANTIZE_COMPONENTS` quantizes both the image backbone and the text encoder by default; set it to `['transformer']` to test the image backbone alone.

Reference image source: `USE_BAKED_IMAGE=True` uses the built-in photo; `ALSO_UPLOAD=True` additionally opens an upload dialog, and uploaded images join the experiment together with the built-in photo (images uploaded earlier in the same runtime are kept and also included). At least one of the two must be on.
The built-in photo is a 1254×1254 square, which becomes 4096 reference tokens after scaling to `REF_MAX_SIDE=1024`; setting `REF_MAX_SIDE` to 768 brings it down to 2304, which noticeably reduces DiT compute but changes the reference detail.

`INT4_BACKEND`: the default `dequant_bf16` keeps INT4 weights packed at 4 bits in GPU memory and expands each layer to BF16 before running cuBLAS; speed should be close to INT8. `quanto_tinygemm` is the previous version's path (about 6x slower per image), kept only to reproduce the comparison. Both paths use exactly the same int4 values, scales and zeros.

In [ ]:
#@title 2. Custom prompt and generation settings
import json, hashlib, time, shutil, gc
from datetime import datetime

PROMPT = "Make the person in the photo dance on a stage, full body in frame, keeping the original facial features and hairstyle, photorealistic photo." #@param {type:"string"}
# Leave empty to test only the prompt above. Batch example:
# EXTRA_PROMPTS = ["Change the background of the photo to a sunset at the beach.", "Change the outfit to a red jacket. Keep the original pose."]
EXTRA_PROMPTS = []
PRECISION_MODE = "bf16,int8,int4" #@param ["bf16,int8,int4", "bf16", "int8", "int4"]
USE_BAKED_IMAGE = True #@param {type:"boolean"}
ALSO_UPLOAD = False #@param {type:"boolean"}
INT4_BACKEND = "dequant_bf16" #@param ["dequant_bf16", "quanto_tinygemm"]

MODEL_ID = 'black-forest-labs/FLUX.2-klein-4B'
MODEL_REVISION = 'main'
LOCAL_MODEL_PATH = ''                         # optional: an existing full Diffusers model directory
PRECISIONS = PRECISION_MODE.split(',')
WIDTH, HEIGHT = 768, 1152                     # 512,512 for a quick test; landscape sizes also work
REF_MAX_SIDE = 1024
SEED = 42
WARMUP_RUNS = 1
REPEATS = 1
QUANTIZE_COMPONENTS = ['transformer', 'text_encoder']
EXCLUDE = {'transformer': ['proj_out'], 'text_encoder': ['lm_head']}
RUN_ID_METRICS = False                       # only meaningful for portraits; independent of generation, keep False for object edits

if not isinstance(PROMPT, str) or not PROMPT.strip():
    raise ValueError('Enter a complete prompt in PROMPT.')
if not isinstance(EXTRA_PROMPTS, list) or any(not isinstance(p,str) or not p.strip() for p in EXTRA_PROMPTS):
    raise ValueError('EXTRA_PROMPTS must be a list of non-empty strings, or an empty list [].')
if INT4_BACKEND not in ('dequant_bf16', 'quanto_tinygemm'):
    raise ValueError('INT4_BACKEND must be dequant_bf16 or quanto_tinygemm.')
if not (USE_BAKED_IMAGE or ALSO_UPLOAD):
    raise ValueError('Turn on at least one of USE_BAKED_IMAGE and ALSO_UPLOAD, otherwise there is no reference image.')
PROMPTS = [PROMPT, *EXTRA_PROMPTS]             # kept verbatim: no cleaning, translation or appended text
PROMPT_STYLE = 'literal'
FACE_DIR = ROOT / 'images_klein_custom'
ASSET_DIR = ROOT / 'klein_assets'
RUN_NAME = datetime.now().strftime('klein_custom_%Y%m%d_%H%M%S_%f')
OUT = ROOT / RUN_NAME
CODE_DIR = OUT / 'code'
for folder in (FACE_DIR, ASSET_DIR, OUT / 'inputs', CODE_DIR):
    folder.mkdir(parents=True, exist_ok=True)
assert WIDTH > 0 and HEIGHT > 0 and WIDTH % 16 == 0 and HEIGHT % 16 == 0
assert REF_MAX_SIDE >= 16
assert WARMUP_RUNS >= 1 and REPEATS >= 1
assert PRECISIONS and set(PRECISIONS) <= {'bf16', 'int8', 'int4'}
assert len(PRECISIONS) == len(set(PRECISIONS))
assert QUANTIZE_COMPONENTS and set(QUANTIZE_COMPONENTS) <= {'transformer', 'text_encoder'}
print('Output folder:', OUT)
print('Precisions:', PRECISIONS, '| output:', (WIDTH, HEIGHT), '| 4 steps | BF16 activations | INT4 path:', INT4_BACKEND)
for i, prompt in enumerate(PROMPTS, 1):
    print(f'\nPrompt {i} (passed verbatim):\n{prompt}')

## 3. Write the self-contained experiment code
The cells below only create the Python modules this notebook uses; the original `f2b_core.py` / `f2b_schemes.py` are not needed.
Each precision runs in its own process, so models or CUDA cache left over from the previous group cannot affect the memory statistics.

In [ ]:
# klein_prompts.py
module_source = r'''"""Reference preprocessing and literal prompt manifest. No prompt rewriting."""

import numpy as np

from PIL import Image, ImageOps

def load_rgb(src, max_side: int = 1600) -> np.ndarray:
    """Load an image (path or PIL), apply EXIF rotation, return HxWx3 uint8 RGB."""
    im = src if isinstance(src, Image.Image) else Image.open(src)
    im = ImageOps.exif_transpose(im).convert("RGB")
    if max(im.size) > max_side:
        s = max_side / max(im.size)
        im = im.resize((round(im.width * s), round(im.height * s)), Image.LANCZOS)
    return np.array(im)

def fit_max_side(rgb: np.ndarray, max_side: int) -> np.ndarray:
    h, w = rgb.shape[:2]
    s = max_side / max(h, w)
    if s >= 1:
        return rgb
    import cv2
    return cv2.resize(rgb, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)



def make_cases(image_paths, input_dir, prompts, seed, reference_max_side):
    from pathlib import Path
    import hashlib
    if not prompts or any(not isinstance(p, str) or not p.strip() for p in prompts):
        raise ValueError('At least one nonempty literal prompt is required.')
    input_dir = Path(input_dir)
    input_dir.mkdir(parents=True, exist_ok=True)
    cases = []
    for fi, path in enumerate(image_paths):
        path = Path(path)
        photo = load_rgb(path, max_side=1600)
        ref = Image.fromarray(fit_max_side(photo, reference_max_side))
        fid = f'image{fi:02d}'
        photo_path = input_dir / f'{fid}_photo.png'
        ref_path = input_dir / f'{fid}_reference.png'
        Image.fromarray(photo).save(photo_path)
        ref.save(ref_path)
        digest = hashlib.sha256(ref_path.read_bytes()).hexdigest()
        for ii, prompt in enumerate(prompts):
            cases.append(dict(case_id=f'{fid}_p{ii:02d}_custom', face=path.name,
                              instruction=prompt, prompt=prompt, lang='custom',
                              prompt_label=f'Prompt {ii+1}', seed=seed,
                              photo_path=str(photo_path), reference_path=str(ref_path),
                              reference_sha256=digest, reference_size=list(ref.size)))
    return cases
'''
(CODE_DIR / 'klein_prompts.py').write_text(module_source, encoding='utf-8')
print('Ready:', 'klein_prompts.py')

In [ ]:
# klein_w4a16.py: fast INT4 path (4-bit packed storage; each layer is dequantized to BF16, then cuBLAS)
module_source = r'''"""INT4 weight storage with BF16 cuBLAS compute (W4A16: dequantize, then GEMM).

Why this exists: Quanto's qint4 path on CUDA sends every matmul through tinygemm
(torch._weight_int4pack_mm), a kernel built for batch-1 LLM decoding. Each thread block computes
only a 16x8 output tile, so with ~8k tokens per call the weights are re-read and re-dequantized
hundreds of times and the linear layers ran ~12x slower than cuBLAS.

What this does instead: weights stay packed at 4 bits (same bytes per parameter as Quanto's int4).
In every forward pass one layer is expanded to BF16 with three elementwise kernels and the matmul
goes to the normal cuBLAS kernel through F.linear.

Numerics: the int4 codes, scales and shifts come from optimum-quanto 0.2.7's own MaxOptimizer and
quantize_affine, so they are bit-identical to Quanto's qint4. For group-128 layers the BF16 weight
is rebuilt with the arithmetic tinygemm applies on CUDA, bf16(fma(q - 8, scale, zero)) with
Quanto's zero = bf16(8 * scale - shift). Layers Quanto keeps on its generic path (in_features <= 128,
e.g. the DiT x_embedder) use Quanto's generic formula bf16(bf16(scale * q) - shift).
"""
import fnmatch
import time

import torch
import torch.nn.functional as F

BACKEND = 'dequant_bf16'
KERNEL_NAME = 'W4 storage -> BF16 dequant + cuBLAS'


def quanto_group_size(in_features):
    """Group-size rule of optimum.quanto QModuleMixin (0.2.7) for qint4 weights; None = per row."""
    group_size = 128
    if in_features > group_size:
        while in_features % group_size != 0 and group_size > 32:
            group_size -= 32
        if in_features % group_size == 0:
            return group_size
    return None


class W4A16Linear(torch.nn.Module):
    """Drop-in replacement for a BF16 nn.Linear: 4-bit packed weight, BF16 compute."""

    def __init__(self, linear):
        super().__init__()
        from optimum.quanto import MaxOptimizer, qint4

        w = linear.weight.detach()
        if w.dtype != torch.bfloat16 or w.dim() != 2:
            raise TypeError(f'W4A16Linear expects a 2-D bfloat16 weight, got {w.dtype} {tuple(w.shape)}')
        n, k = w.shape
        group_size = quanto_group_size(k)
        g = k if group_size is None else group_size
        if g % 2:
            raise ValueError(f'in_features={k}: a group of {g} values cannot be packed two per byte')
        self.in_features, self.out_features, self.group_size = k, n, g
        # Quanto picks tinygemm for qint4 + bf16 + group 128 on sm80+; everything else is dequantized generically.
        self.tinygemm_formula = group_size == 128

        scale, shift = MaxOptimizer()(w, qtype=qint4, axis=0, group_size=group_size)
        q = torch.ops.quanto.quantize_affine(w, bits=4, axis=0, group_size=group_size, scale=scale, shift=shift)
        groups = k // g
        q = q.reshape(n, groups, 2, g // 2)            # [row, group, half of group, position]
        scale = scale.reshape(n, groups, 1, 1)
        shift = shift.reshape(n, groups, 1, 1)
        if self.tinygemm_formula:
            codes = q ^ 8                               # 4-bit two's complement of q - 8
            zero = (-shift) + 2 ** (qint4.bits - 1) * scale   # Quanto's tinygemm zero, same bf16 op order
            nibble_shifts = torch.tensor([4, 0], dtype=torch.int8, device=w.device)
        else:
            codes = q
            zero = shift
            nibble_shifts = torch.tensor([0, 4], dtype=torch.uint8, device=w.device)
        # One byte holds element j (low nibble) and element j + g/2 (high nibble) of the same group.
        self.register_buffer('packed', ((codes[:, :, 1] << 4) | codes[:, :, 0]).contiguous())
        self.register_buffer('scale', scale.contiguous())
        self.register_buffer('zero', zero.contiguous())
        self.register_buffer('nibble_shifts', nibble_shifts.view(1, 1, 2, 1), persistent=False)
        if linear.bias is not None:
            self.bias = torch.nn.Parameter(linear.bias.detach(), requires_grad=False)
        else:
            self.register_parameter('bias', None)
        self.train(linear.training)

    def dequantize(self):
        if self.tinygemm_formula:
            t = torch.bitwise_left_shift(self.packed.view(torch.int8).unsqueeze(2), self.nibble_shifts)
            t.bitwise_right_shift_(4)                   # arithmetic shift: signed q - 8 in [-8, 7]
            w = torch.addcmul(self.zero, t, self.scale) # fp32 math, one rounding to bf16 (= tinygemm __hfma2)
        else:
            t = torch.bitwise_right_shift(self.packed.unsqueeze(2), self.nibble_shifts)
            t.bitwise_and_(15)
            w = self.scale * t                          # Quanto generic: bf16(scale * q) ...
            w.sub_(self.zero)                           # ... then bf16(. - shift)
        return w.view(self.out_features, self.in_features)

    def forward(self, x):
        return F.linear(x, self.dequantize(), self.bias)

    # ---- validation helpers (slow, independent of the fast path) ----
    @torch.no_grad()
    def unpacked_codes(self):
        """The Quanto int4 codes q in [0, 15], shape [out, in]."""
        p = self.packed.to(torch.int32)
        codes = torch.stack((p & 15, p >> 4), dim=2)
        if self.tinygemm_formula:
            codes = codes ^ 8
        return codes.reshape(self.out_features, self.in_features).to(torch.uint8)

    @torch.no_grad()
    def reference_dequantize(self):
        q = self.unpacked_codes().reshape(self.packed.shape[0], self.packed.shape[1], 2, -1).float()
        if self.tinygemm_formula:
            w = ((q - 8) * self.scale.float() + self.zero.float()).to(torch.bfloat16)
        else:
            w = (q * self.scale.float()).to(torch.bfloat16)
            w = (w.float() - self.zero.float()).to(torch.bfloat16)
        return w.reshape(self.out_features, self.in_features)

    def extra_repr(self):
        formula = 'tinygemm' if self.tinygemm_formula else 'quanto-generic'
        return (f'in_features={self.in_features}, out_features={self.out_features}, '
                f'group_size={self.group_size}, formula={formula}, bias={self.bias is not None}')


def quantize_module_w4a16(model, exclude, component_name):
    """Replace every nn.Linear not matching `exclude` with W4A16Linear; fail loudly on any leftover."""
    targets = [name for name, m in model.named_modules()
               if isinstance(m, torch.nn.Linear) and not any(fnmatch.fnmatchcase(name, p) for p in exclude)]
    if not targets:
        raise RuntimeError(f'{component_name}: no Linear layers selected; refusing an unquantized result.')
    elements = 0
    for name in targets:
        parent_name, _, child = name.rpartition('.')
        parent = model.get_submodule(parent_name) if parent_name else model
        linear = getattr(parent, child)
        elements += linear.weight.numel()
        setattr(parent, child, W4A16Linear(linear))
        del linear
    actual = {name: m for name, m in model.named_modules() if isinstance(m, W4A16Linear)}
    if set(actual) != set(targets):
        raise RuntimeError(f'{component_name}: requested {len(targets)} layers, converted {len(actual)}.')
    leftover = [n for n, m in model.named_modules() if n in set(targets) and type(m) is torch.nn.Linear]
    if leftover:
        raise RuntimeError(f'{component_name}: still BF16 after conversion: {leftover[:5]}')
    layer_map, formulas = {}, {'tinygemm': 0, 'quanto-generic': 0}
    for name, m in actual.items():
        if m.packed.dtype != torch.uint8 or m.packed.numel() * 2 != m.in_features * m.out_features:
            raise RuntimeError(f'{component_name}.{name}: packed weight has the wrong size or dtype.')
        if hasattr(m, 'weight'):
            raise RuntimeError(f'{component_name}.{name}: a full-precision weight is still attached.')
        formula = 'tinygemm' if m.tinygemm_formula else 'quanto-generic'
        formulas[formula] += 1
        layer_map[name] = dict(weights='qint4', group_size=m.group_size, dequant_formula=formula)
    return dict(component=component_name, backend=BACKEND, kernel=KERNEL_NAME,
                quantized_linear_layers=len(actual), quantized_weight_elements=elements,
                excluded_patterns=exclude, dequant_formulas=formulas, quantization_map=layer_map)


@torch.no_grad()
def smoke_test(device='cuda', benchmark_shape=(8064, 3072, 27648)):
    """Exactness checks on small layers, then a speed check on the largest DiT matmul shape."""
    from optimum.quanto import MaxOptimizer, qint4

    torch.manual_seed(0)
    mismatches = 0
    for k, n in ((256, 256), (128, 3072), (3072, 640)):
        linear = torch.nn.Linear(k, n, bias=False).to(device=device, dtype=torch.bfloat16)
        w0 = linear.weight.detach().clone()
        qmod = W4A16Linear(linear)
        gs = quanto_group_size(k)
        scale, shift = MaxOptimizer()(w0, qtype=qint4, axis=0, group_size=gs)
        q_ref = torch.ops.quanto.quantize_affine(w0, bits=4, axis=0, group_size=gs, scale=scale, shift=shift)
        if not torch.equal(qmod.unpacked_codes(), q_ref.reshape(n, k)):
            raise RuntimeError(f'W4A16 smoke test {k}->{n}: packed codes differ from Quanto quantize_affine.')
        w = qmod.dequantize()
        ref = qmod.reference_dequantize()
        diff = (w.float() - ref.float()).abs()
        if (diff > ref.float().abs() * 2 ** -7).any().item():   # more than ~1 bf16 ulp means a real bug
            raise RuntimeError(f'W4A16 smoke test {k}->{n}: fast dequantization disagrees with the reference.')
        mismatches += int((diff > 0).sum().item())
        rel = ((w.float() - w0.float()).norm() / w0.float().norm()).item()
        if not rel < 0.2:
            raise RuntimeError(f'W4A16 smoke test {k}->{n}: relative weight error {rel:.3f} is too large.')
        x = torch.randn(37, k, device=device, dtype=torch.bfloat16)
        y = qmod(x).float()
        y_ref = F.linear(x, w).float()
        if not torch.isfinite(y).all().item() or (y - y_ref).abs().max().item() > 1e-2 * y_ref.abs().max().item():
            raise RuntimeError(f'W4A16 smoke test {k}->{n}: forward does not match F.linear on the dequantized weight.')
        del linear, qmod, w0, w, ref, x, y, y_ref
    result = dict(checks='codes == Quanto quantize_affine; dequant == reference; forward == F.linear',
                  passed=True, dequant_bitwise_mismatches=mismatches)
    if device == 'cuda' and benchmark_shape:
        m, k, n = benchmark_shape
        x = torch.randn(m, k, device=device, dtype=torch.bfloat16)
        linear = torch.nn.Linear(k, n, bias=False).to(device=device, dtype=torch.bfloat16)

        def bench(fn, iters=10):
            for _ in range(3):
                fn()
            torch.cuda.synchronize()
            start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
            start.record()
            for _ in range(iters):
                fn()
            end.record()
            torch.cuda.synchronize()
            return start.elapsed_time(end) / iters

        bf16_ms = bench(lambda: linear(x))
        qmod = W4A16Linear(linear)
        del linear
        w4_ms = bench(lambda: qmod(x))
        deq_ms = bench(qmod.dequantize)
        result.update(shape=[m, k, n], bf16_ms=bf16_ms, w4a16_ms=w4_ms, dequant_only_ms=deq_ms,
                      slowdown_vs_bf16=w4_ms / bf16_ms)
        del x, qmod
        torch.cuda.empty_cache()
    return result
'''
(CODE_DIR / 'klein_w4a16.py').write_text(module_source, encoding='utf-8')
print('Ready:', 'klein_w4a16.py')

In [ ]:
# klein_quant_worker.py
module_source = r'''"""One CUDA process per precision. Only S2c; no SD1.5, masks, warps or face swapping."""
import argparse
import gc
import hashlib
import importlib.metadata as metadata
import json
import os
from pathlib import Path
import platform
import sys
import time
import traceback

import numpy as np
from PIL import Image


def write_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    tmp.replace(path)


def storage_bytes(model):
    """Count actual unique tensor storage, including packed weights and quantization metadata.

    Quanto state_dict serialization normally exposes plain tensors. Recurse through tensor
    subclasses as well; never estimate a packed tensor as numel * its apparent BF16 dtype.
    """
    import torch
    seen, total = set(), 0

    def visit(t):
        nonlocal total
        if not isinstance(t, torch.Tensor):
            return
        if type(t) not in (torch.Tensor, torch.nn.Parameter) and hasattr(t, '__tensor_flatten__'):
            names, _ = t.__tensor_flatten__()
            for name in names:
                visit(getattr(t, name))
            return
        s = t.untyped_storage()
        key = (str(t.device), s.data_ptr(), s.nbytes())
        if key not in seen:
            seen.add(key)
            total += s.nbytes()

    for t in model.state_dict().values():
        visit(t)
    return total


def quantize_component(model, precision, exclude, component_name, int4_backend='quanto_tinygemm'):
    if precision == 'int4' and int4_backend == 'dequant_bf16':
        # Same Quanto qint4 numerics; 4-bit packed storage + BF16 dequant + cuBLAS (klein_w4a16.py).
        from klein_w4a16 import quantize_module_w4a16
        return quantize_module_w4a16(model, exclude, component_name)
    import fnmatch
    import torch
    from optimum.quanto import freeze, qint4, qint8, quantization_map, quantize

    # Explicit allowlist: Linear weights only; embeddings, normalization and convolutions stay BF16.
    targets = [name for name, m in model.named_modules()
               if isinstance(m, torch.nn.Linear)
               and not any(fnmatch.fnmatchcase(name, p) for p in exclude)]
    if not targets:
        raise RuntimeError(f'{component_name}: no Linear layers selected; refusing an unquantized result.')
    original_elements = {name: m.weight.numel() for name, m in model.named_modules() if name in targets}
    qtype = qint8 if precision == 'int8' else qint4
    quantize(model, weights=qtype, activations=None, include=targets)
    freeze(model)  # Essential: replace float weights with packed integer weights, not fake quantization.
    actual = quantization_map(model)
    if set(actual) != set(targets):
        raise RuntimeError(f'{component_name}: requested {len(targets)} layers, quantized {len(actual)}.')
    for name, settings in actual.items():
        if settings['weights'] != qtype.name or settings['activations'] != 'none':
            raise RuntimeError(f'Unexpected quantization at {component_name}.{name}: {settings}')
        w = model.get_submodule(name).weight
        if getattr(w, 'qtype', None) != qtype:
            raise RuntimeError(f'{component_name}.{name}: weight was not frozen to {precision}.')
    return dict(component=component_name, quantized_linear_layers=len(actual),
                quantized_weight_elements=sum(original_elements.values()),
                excluded_patterns=exclude, quantization_map=actual)


def kernel_label(precision, int4_backend):
    if precision == 'bf16':
        return 'BF16 weights + cuBLAS'
    if precision == 'int8':
        return 'Quanto qint8 -> BF16 dequant + cuBLAS'
    if int4_backend == 'dequant_bf16':
        return 'qint4 packed -> BF16 dequant + cuBLAS'
    return 'Quanto qint4 -> tinygemm (torch._weight_int4pack_mm)'


def quantization_smoke_test(precision, int4_backend='quanto_tinygemm'):
    """Fail quickly on incompatible Torch/Quanto/CUDA installations, before loading the 8B pipeline."""
    import torch
    if precision == 'bf16':
        return None
    if precision == 'int4' and int4_backend == 'dequant_bf16':
        from klein_w4a16 import smoke_test
        result = smoke_test(device='cuda')
        print(f"[int4] W4A16 checks passed (bitwise dequant mismatches: {result['dequant_bitwise_mismatches']}). "
              f"Largest DiT matmul {result['shape']}: BF16 {result['bf16_ms']:.2f} ms, "
              f"INT4 dequant+cuBLAS {result['w4a16_ms']:.2f} ms (dequant alone {result['dequant_only_ms']:.2f} ms)",
              flush=True)
        gc.collect()
        torch.cuda.empty_cache()
        return result
    # The small square kernel alone did not cover the klein x_embedder fallback path.
    for in_features, out_features in [(256, 256), (128, 3072)]:
        m = torch.nn.Sequential(torch.nn.Linear(in_features, out_features)).to(device='cuda', dtype=torch.bfloat16)
        m.eval().requires_grad_(False)
        quantize_component(m, precision, [], 'smoke_test')
        with torch.no_grad():
            y = m(torch.randn(2, in_features, device='cuda', dtype=torch.bfloat16))
            if not torch.isfinite(y).all().item():
                raise FloatingPointError('Quantized Linear smoke test produced NaN/Inf.')
        del m, y
    gc.collect()
    torch.cuda.empty_cache()
    return dict(passed=True)


def run(cfg, precision, mode_dir):
    import torch
    from diffusers import Flux2KleinPipeline

    if not torch.cuda.is_available():
        raise RuntimeError('CUDA GPU required. In Colab select an A100 GPU runtime.')
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError('This experiment preserves BF16: select A100 or another BF16-capable CUDA GPU.')
    torch.cuda.set_device(0)
    torch.set_grad_enabled(False)
    torch.manual_seed(cfg['seed'])
    torch.cuda.manual_seed_all(cfg['seed'])
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

    env = dict(python=platform.python_version(), gpu=torch.cuda.get_device_name(0),
               total_vram_gib=torch.cuda.get_device_properties(0).total_memory / 2**30,
               cuda=torch.version.cuda, compute_capability=list(torch.cuda.get_device_capability(0)),
               packages={n: metadata.version(n) for n in
                         ['torch', 'diffusers', 'transformers', 'optimum-quanto', 'accelerate', 'safetensors']})
    write_json(mode_dir / 'environment.json', env)
    print(f'[{precision}] GPU: {env["gpu"]}; package versions: {env["packages"]}', flush=True)
    int4_backend = cfg.get('int4_backend', 'quanto_tinygemm')
    kernel = kernel_label(precision, int4_backend)
    print(f'[{precision}] matmul path: {kernel}', flush=True)
    kernel_check = quantization_smoke_test(precision, int4_backend)

    torch.cuda.reset_peak_memory_stats()
    print(f'[{precision}] Loading the same model snapshot in BF16...', flush=True)
    t0 = time.perf_counter()
    pipe = Flux2KleinPipeline.from_pretrained(
        cfg['model_path'], torch_dtype=torch.bfloat16, local_files_only=True,
        low_cpu_mem_usage=True,
    ).to('cuda')
    pipe.set_progress_bar_config(disable=True)
    components = {name: getattr(pipe, name) for name in ['transformer', 'text_encoder', 'vae']}
    for model in components.values():
        model.eval().requires_grad_(False)
    if not bool(pipe.config.get('is_distilled', False)):
        raise RuntimeError('Expected the distilled klein 4B model, not klein-base-4B.')
    torch.cuda.synchronize()
    load_s = time.perf_counter() - t0
    original_params = {name: sum(p.numel() for p in model.parameters()) for name, model in components.items()}

    audits = []
    t0 = time.perf_counter()
    if precision != 'bf16':
        for component_name in cfg['quantize_components']:
            print(f'[{precision}] Quantizing {component_name} Linear weights...', flush=True)
            audits.append(quantize_component(components[component_name], precision,
                                            cfg['exclude'][component_name], component_name,
                                            int4_backend))
    torch.cuda.synchronize()
    quantize_s = time.perf_counter() - t0 if precision != 'bf16' else 0.0
    load_quant_peak_gib = torch.cuda.max_memory_allocated() / 2**30
    gc.collect()
    torch.cuda.empty_cache()

    component_info = {}
    for name, model in components.items():
        info = dict(original_parameters=original_params[name])
        try:
            info['tensor_storage_bytes'] = storage_bytes(model)
            info['tensor_storage_mib'] = info['tensor_storage_bytes'] / 2**20
        except Exception as exc:
            info['tensor_storage_bytes'] = None
            info['storage_measurement_error'] = str(exc)
        component_info[name] = info
    sizes = [x['tensor_storage_bytes'] for x in component_info.values()]
    stats = dict(precision=precision, compute_dtype='bfloat16', grad_mode='no_grad', kernel=kernel,
                 int4_backend=int4_backend if precision == 'int4' else None, kernel_check=kernel_check,
                 quantization_backend=None if precision == 'bf16' else 'optimum-quanto numerics',
                 quantize_components=cfg['quantize_components'] if precision != 'bf16' else [],
                 load_s=load_s, quantize_s=quantize_s, load_quant_peak_allocated_gib=load_quant_peak_gib,
                 resident_allocated_gib=torch.cuda.memory_allocated() / 2**30,
                 tensor_storage_mib=sum(sizes) / 2**20 if all(x is not None for x in sizes) else None,
                 components=component_info, audit=audits, scheduler_class=type(pipe.scheduler).__name__,
                 scheduler_config=dict(pipe.scheduler.config), warmup_times_s=[])
    write_json(mode_dir / 'model_stats.json', stats)

    # References were normalized once in the parent, reused byte-for-byte by every precision.
    refs = {}
    for case in cfg['cases']:
        path = case['reference_path']
        if path not in refs:
            data = Path(path).read_bytes()
            if hashlib.sha256(data).hexdigest() != case['reference_sha256']:
                raise RuntimeError('Reference file changed after experiment configuration was written.')
            with Image.open(path) as im:
                refs[path] = im.convert('RGB').copy()

    def generate(case, check_latents=False):
        step_count = 0

        def on_step(pipeline, step, timestep, callback_kwargs):
            nonlocal step_count
            step_count += 1
            if check_latents and not torch.isfinite(callback_kwargs['latents']).all().item():
                raise FloatingPointError(f'Non-finite latents at step {step}. No dtype fallback applied.')
            return callback_kwargs

        # Same CUDA generator, seed and call sequence as the original S2c implementation.
        generator = torch.Generator(device='cuda').manual_seed(case['seed'])
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        before = torch.cuda.memory_allocated()
        t0 = time.perf_counter()
        with torch.no_grad():
            array = pipe(image=[refs[case['reference_path']]], prompt=case['prompt'],
                         width=cfg['width'], height=cfg['height'], num_inference_steps=4,
                         guidance_scale=1.0, generator=generator, max_sequence_length=512,
                         callback_on_step_end=on_step, callback_on_step_end_tensor_inputs=['latents'],
                         output_type='np').images[0]
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0
        mem = dict(peak_allocated_gib=torch.cuda.max_memory_allocated() / 2**30,
                   peak_reserved_gib=torch.cuda.max_memory_reserved() / 2**30,
                   incremental_peak_gib=(torch.cuda.max_memory_allocated() - before) / 2**30)
        if step_count != 4:
            raise RuntimeError(f'Expected exactly 4 steps, observed {step_count}.')
        array = np.asarray(array)
        if array.shape != (cfg['height'], cfg['width'], 3) or not np.isfinite(array).all():
            raise FloatingPointError(f'Invalid image output: shape={array.shape}, NaN/Inf possible.')
        image = Image.fromarray((np.clip(array, 0, 1) * 255).round().astype(np.uint8))
        return image, elapsed, mem, step_count, float(array.std())

    print(f'[{precision}] Warm-up ({cfg["warmup_runs"]} full 4-step calls; not in benchmark)...', flush=True)
    for _ in range(cfg['warmup_runs']):
        image, seconds, _, _, _ = generate(cfg['cases'][0], check_latents=True)
        stats['warmup_times_s'].append(seconds)
        del image
    write_json(mode_dir / 'model_stats.json', stats)

    rows = []
    for ci, case in enumerate(cfg['cases']):
        for repeat in range(cfg['repeats']):
            row = dict(case_id=case['case_id'], face=case['face'], instruction=case['instruction'],
                       lang=case['lang'], prompt=case['prompt'], precision=precision, seed=case['seed'],
                       repeat=repeat, width=cfg['width'], height=cfg['height'], status='running')
            try:
                image, seconds, mem, steps, std = generate(case)
                name = f'{case["case_id"]}__{precision}__r{repeat}.png'
                image.save(mode_dir / name)  # Disk I/O is excluded from generation_e2e_s.
                row.update(status='ok', image=f'{precision}/{name}', generation_e2e_s=seconds,
                           actual_steps=steps, image_std=std, near_uniform_warning=bool(std < .005), **mem)
                print(f'[{precision}] {ci+1}/{len(cfg["cases"])} {case["case_id"]} r{repeat}: '
                      f'{seconds:.3f} s, peak {mem["peak_allocated_gib"]:.2f} GiB', flush=True)
                del image
            except Exception as exc:
                row.update(status='error', error=repr(exc), traceback=traceback.format_exc())
                print(row['traceback'], flush=True)
                rows.append(row)
                write_json(mode_dir / 'results.json', rows)
                if isinstance(exc, torch.cuda.OutOfMemoryError):
                    raise  # End this process cleanly; do not silently downscale/offload.
                continue
            rows.append(row)
            write_json(mode_dir / 'results.json', rows)
    failures = sum(x['status'] != 'ok' for x in rows)
    return dict(status='complete' if failures == 0 else 'partial', precision=precision,
                successful_images=len(rows)-failures, failed_images=failures)


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', required=True)
    parser.add_argument('--precision', choices=['bf16', 'int8', 'int4'], required=True)
    args = parser.parse_args()
    config = json.loads(Path(args.config).read_text(encoding='utf-8'))
    mode_dir = Path(config['output_dir']) / args.precision
    mode_dir.mkdir(parents=True, exist_ok=True)
    try:
        outcome = run(config, args.precision, mode_dir)
        write_json(mode_dir / 'status.json', outcome)
    except Exception as exc:
        error = traceback.format_exc()
        print(error, file=sys.stderr, flush=True)
        write_json(mode_dir / 'status.json', dict(status='failed', precision=args.precision,
                                                error=repr(exc), traceback=error))
        sys.exit(1)
'''
(CODE_DIR / 'klein_quant_worker.py').write_text(module_source, encoding='utf-8')
print('Ready:', 'klein_quant_worker.py')

In [ ]:
# klein_identity.py
module_source = r'''"""Optional CPU evaluation. ArcFace/alignment copied from original notebook, no generation."""

import os, json, zipfile, shutil, urllib.request

from pathlib import Path

from types import SimpleNamespace

import numpy as np

import cv2

from PIL import Image

ARCFACE_ZIP_URL = "https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip"

def download(url: str, path: str) -> str:
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    tmp = path + ".part"
    urllib.request.urlretrieve(url, tmp)
    os.replace(tmp, path)
    return path

def eye_centers(pts: np.ndarray):
    """(image-left eye centre, image-right eye centre)."""
    a = pts[[33, 133]].mean(0)
    b = pts[[362, 263]].mean(0)
    return (a, b) if a[0] <= b[0] else (b, a)

def similarity_transform(src, dst) -> np.ndarray:
    """Least-squares similarity (Umeyama, no reflection). Returns a 2x3 matrix mapping src->dst."""
    src = np.asarray(src, np.float64)
    dst = np.asarray(dst, np.float64)
    mu_s, mu_d = src.mean(0), dst.mean(0)
    s0, d0 = src - mu_s, dst - mu_d
    cov = d0.T @ s0 / len(src)
    U, S, Vt = np.linalg.svd(cov)
    D = np.eye(2)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        D[1, 1] = -1
    R = U @ D @ Vt
    var_s = (s0 ** 2).sum() / len(src)
    scale = float(np.trace(np.diag(S) @ D) / var_s)
    t = mu_d - scale * R @ mu_s
    return np.hstack([scale * R, t[:, None]]).astype(np.float32)

fc = SimpleNamespace(download=download, eye_centers=eye_centers, similarity_transform=similarity_transform, NOSE_TIP=1, MOUTH_CORNERS=(61,291))

class ArcFace:
    """insightface buffalo_l recognition model (w600k_r50) run with onnxruntime; MediaPipe landmarks for alignment."""

    TEMPLATE = np.array([[38.2946, 51.6963], [73.5318, 51.5014], [56.0252, 71.7366],
                         [41.5493, 92.3655], [70.7299, 92.2041]], np.float32)

    def __init__(self, model_dir="models", zip_url=ARCFACE_ZIP_URL, member="w600k_r50.onnx"):
        import onnxruntime as ort
        path = os.path.join(model_dir, member)
        if not os.path.exists(path):
            zpath = fc.download(zip_url, os.path.join(model_dir, "buffalo_l.zip"))
            with zipfile.ZipFile(zpath) as zf:
                name = next(n for n in zf.namelist() if n.endswith(member))
                with zf.open(name) as src, open(path + ".part", "wb") as dst:
                    shutil.copyfileobj(src, dst)
            os.replace(path + ".part", path)
        self.sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
        self.inp = self.sess.get_inputs()[0].name

    @staticmethod
    def five_points(pts: np.ndarray) -> np.ndarray:
        le, re_ = fc.eye_centers(pts)
        a, b = pts[fc.MOUTH_CORNERS[0]], pts[fc.MOUTH_CORNERS[1]]
        ml, mr = (a, b) if a[0] <= b[0] else (b, a)
        return np.stack([le, re_, pts[fc.NOSE_TIP], ml, mr]).astype(np.float32)

    def align(self, rgb: np.ndarray, pts: np.ndarray, size: int = 112) -> np.ndarray:
        """Same geometry as insightface face_align.norm_crop (112 -> template, 224 -> template x 2)."""
        M = fc.similarity_transform(self.five_points(pts), self.TEMPLATE * (size / 112.0))
        return cv2.warpAffine(rgb, M, (size, size), flags=cv2.INTER_LINEAR, borderValue=0)

    def embed(self, rgb: np.ndarray, pts: np.ndarray) -> np.ndarray:
        crop = self.align(rgb, pts, 112).astype(np.float32)
        x = ((crop - 127.5) / 127.5).transpose(2, 0, 1)[None]          # RGB, as insightface feeds it
        e = self.sess.run(None, {self.inp: x})[0][0].astype(np.float32)
        return e / (np.linalg.norm(e) + 1e-8)


def evaluate(root):
    import mediapipe as mp
    from mediapipe.tasks.python import BaseOptions, vision
    root = Path(root)
    cfg = json.loads((root / 'config.json').read_text(encoding='utf-8'))
    model_dir = Path(cfg['asset_dir']) / 'identity'
    model_dir.mkdir(parents=True, exist_ok=True)
    path = download('https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task',
                    str(model_dir / 'face_landmarker.task'))
    landmarker = vision.FaceLandmarker.create_from_options(
        vision.FaceLandmarkerOptions(base_options=BaseOptions(model_asset_path=path), num_faces=4))
    arcface = ArcFace(str(model_dir))

    def embedding(image_path):
        with Image.open(image_path) as im:
            rgb = np.asarray(im.convert('RGB'))
        h, w = rgb.shape[:2]
        found = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb)))
        points = [np.array([[p.x*w, p.y*h] for p in lm], dtype=np.float32) for lm in found.face_landmarks]
        points.sort(key=lambda pts: np.linalg.norm(eye_centers(pts)[1]-eye_centers(pts)[0]), reverse=True)
        return (arcface.embed(rgb, points[0]) if points else None), len(points)

    ref_cache, scores = {}, {}
    case_lookup = {c['case_id']: c for c in cfg['cases']}
    for mode in cfg['precisions']:
        results = root / mode / 'results.json'
        if not results.is_file():
            continue
        for row in json.loads(results.read_text(encoding='utf-8')):
            if row['status'] != 'ok':
                continue
            try:
                refpath = case_lookup[row['case_id']]['photo_path']
                if refpath not in ref_cache:
                    ref_cache[refpath] = embedding(refpath)
                ref, ref_n = ref_cache[refpath]
                emb, n = embedding(root / row['image'])
                state = 'ok' if ref is not None and emb is not None else 'face_not_found'
                scores[row['image']] = dict(identity_status=state, n_faces=n, reference_n_faces=ref_n,
                                            id_sim=float(np.dot(ref, emb)) if state == 'ok' else None)
            except Exception as exc:
                scores[row['image']] = dict(identity_status='error', identity_error=repr(exc), id_sim=None)
    landmarker.close()
    (root / 'identity.json').write_text(json.dumps(scores, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'Identity evaluation: {len(scores)} outputs; using original w600k_r50 and alignment.')


if __name__ == '__main__':
    import sys
    evaluate(sys.argv[1])
'''
(CODE_DIR / 'klein_identity.py').write_text(module_source, encoding='utf-8')
print('Ready:', 'klein_identity.py')


In [ ]:
# klein_report.py
module_source = r'''"""CPU-only comparison; imports neither torch nor any generation model."""
import html
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont, ImageOps


def read_json(path, default=None):
    return json.loads(Path(path).read_text(encoding='utf-8')) if Path(path).is_file() else default


def build_report(output_dir):
    root = Path(output_dir)
    cfg = read_json(root / 'config.json')
    identity = read_json(root / 'identity.json', {})
    rows, summary, statuses = [], [], []
    for mode in cfg['precisions']:
        mode_rows = read_json(root / mode / 'results.json', [])
        for row in mode_rows:
            row.update(identity.get(row.get('image'), {}))
        stats = read_json(root / mode / 'model_stats.json', {})
        status = read_json(root / mode / 'status.json', {'status': 'missing'})
        statuses.append(dict(precision=mode, **{k: v for k, v in status.items() if k != 'precision'}))
        rows.extend(mode_rows)
        good = [r for r in mode_rows if r['status'] == 'ok']
        summary.append(dict(precision=mode, status=status['status'], kernel=stats.get('kernel'), n_ok=len(good),
                            n_expected=len(cfg['cases']) * cfg['repeats'],
                            generation_median_s=np.median([r['generation_e2e_s'] for r in good]) if good else None,
                            peak_allocated_gib=max([r['peak_allocated_gib'] for r in good], default=None),
                            peak_reserved_gib=max([r['peak_reserved_gib'] for r in good], default=None),
                            tensor_storage_mib=stats.get('tensor_storage_mib'),
                            resident_allocated_gib=stats.get('resident_allocated_gib'),
                            load_s=stats.get('load_s'), quantize_s=stats.get('quantize_s'),
                            id_sim_mean=np.mean([r['id_sim'] for r in good if r.get('id_sim') is not None])
                            if any(r.get('id_sim') is not None for r in good) else None))
    baseline = {(r['case_id'], r['repeat']): r for r in rows if r['precision'] == 'bf16' and r['status'] == 'ok'}
    for row in rows:
        if row['status'] != 'ok':
            continue
        reference = baseline.get((row['case_id'], row['repeat']))
        if reference is None:
            continue
        a = np.asarray(Image.open(root / row['image']).convert('RGB'), dtype=np.float32) / 255
        b = np.asarray(Image.open(root / reference['image']).convert('RGB'), dtype=np.float32) / 255
        mse = float(np.mean((a-b)**2))
        row['mae_vs_bf16'] = float(np.mean(np.abs(a-b)))
        row['psnr_vs_bf16_db'] = -10 * math.log10(mse) if mse > 0 else float('inf')
        row['speedup_vs_bf16'] = reference['generation_e2e_s'] / row['generation_e2e_s']
    df, sf = pd.DataFrame(rows), pd.DataFrame(summary)
    # Speed comparison is paired by case and repeat; do not compare different surviving subsets.
    for i, mode in enumerate(cfg['precisions']):
        ratios = [r['speedup_vs_bf16'] for r in rows if r['precision'] == mode and 'speedup_vs_bf16' in r]
        sf.loc[i, 'paired_speedup_median'] = np.median(ratios) if ratios else np.nan
        sf.loc[i, 'paired_comparisons'] = len(ratios)
    df.to_csv(root / 'metrics.csv', index=False)
    sf.to_csv(root / 'summary.csv', index=False)
    (root / 'statuses.json').write_text(json.dumps(statuses, ensure_ascii=False, indent=2), encoding='utf-8')

    lookup = {(r['case_id'], r['precision']): r for r in rows if r['repeat'] == 0}
    cards = []
    for case in cfg['cases']:
        cells = [f'<figure><figcaption>Reference</figcaption><img src="inputs/{Path(case["reference_path"]).name}"></figure>']
        for mode in cfg['precisions']:
            r = lookup.get((case['case_id'], mode))
            if r and r['status'] == 'ok':
                text = f'{mode.upper()} · {r["generation_e2e_s"]:.2f} s · {r["peak_allocated_gib"]:.2f} GiB'
                if r.get('id_sim') is not None:
                    text += f' · ID {r["id_sim"]:.3f}'
                cells.append(f'<figure><figcaption>{text}</figcaption><img src="{html.escape(r["image"])}"></figure>')
            else:
                err = r.get('error', 'No successful result — see log/status.json') if r else 'No successful result — see log/status.json'
                cells.append(f'<figure><figcaption>{mode.upper()}</figcaption><pre>{html.escape(err)}</pre></figure>')
        cards.append(f'<article><h2>{html.escape(case["face"])} · {html.escape(case.get("prompt_label", case["case_id"]))}</h2>'
                     f'<p class="prompt"><b>Actual prompt:</b> {html.escape(case["prompt"])}</p>'
                     f'<div class="images">{"".join(cells)}</div></article>')
        # PNG has short English labels; full Chinese/English prompts remain legible in HTML.
        tw, th, bar = 256, 384, 48
        grid = Image.new('RGB', (tw * (1 + len(cfg['precisions'])), th + bar), 'white')
        draw = ImageDraw.Draw(grid)
        try:
            font = ImageFont.truetype('DejaVuSans.ttf', 14)
        except OSError:
            font = ImageFont.load_default()
        specs = [('REFERENCE', Path(case['reference_path']))]
        for mode in cfg['precisions']:
            r = lookup.get((case['case_id'], mode))
            specs.append((mode.upper() + (f' | {r["generation_e2e_s"]:.2f}s' if r and r['status'] == 'ok' else ' | FAILED'),
                          root / r['image'] if r and r['status'] == 'ok' else None))
        for j, (label, path) in enumerate(specs):
            draw.text((j*tw+8, 12), label, font=font, fill='black')
            if path is not None:
                with Image.open(path) as im:
                    thumb = ImageOps.contain(im.convert('RGB'), (tw, th), Image.Resampling.LANCZOS)
                grid.paste(thumb, (j*tw+(tw-thumb.width)//2, bar+(th-thumb.height)//2))
        grid.save(root / f'grid_{case["case_id"]}.png')
    int4_note = ('INT4 uses Quanto qint4 numerics, stored packed at 4 bits and dequantized to BF16 per layer for cuBLAS. '
                 if cfg.get('int4_backend', 'quanto_tinygemm') == 'dequant_bf16'
                 else 'INT4 runs through Quanto tinygemm (torch._weight_int4pack_mm). ')
    source_note = ('BF16 baseline; INT8/INT4 are integer weight-only quantization. ' + int4_note +
                   'Selected Linear layers in ' + ', '.join(cfg['quantize_components']) +
                   '; activations, VAE, embeddings/norms and excluded layers stay BF16. '
                   'Literal user prompts, with no translation or appended template. '
                   '4 steps; CFG=1; identical seeds/references/prompts; no offload/compile; no_grad for all modes. '
                   'E2E includes text encoding, image encoding, denoising and decode; excludes download/load/quantization/warm-up/file I/O. '
                   'Tensor storage includes packed data and scales, not total runtime memory. '
                   'Pixel differences measure drift from BF16, not identity or overall quality. These are CUDA timings, not phone timings.')
    page = '<!doctype html><html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">'
    page += '<title>Klein custom prompts quantization comparison</title><style>body{font:15px system-ui;background:#f3f5f9;color:#172033;margin:24px}article,header{background:white;padding:20px;border-radius:12px;margin-bottom:20px}.images{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:12px}figure{margin:0}img{width:100%;height:auto}figcaption{padding:10px 0;font-weight:600}table{border-collapse:collapse;display:block;overflow:auto}th,td{padding:8px;border:1px solid #ddd}pre{white-space:pre-wrap;overflow-wrap:anywhere}.prompt{line-height:1.7;background:#eef2f8;padding:12px}</style>'
    page += f'<header><h1>FLUX.2 klein 4B · custom prompts</h1><p>{html.escape(source_note)}</p>{sf.round(3).to_html(index=False,escape=True)}'
    page += f'<details><summary>Run status / errors</summary><pre>{html.escape(json.dumps(statuses,ensure_ascii=False,indent=2))}</pre></details></header>'
    page += ''.join(cards) + '</html>'
    (root / 'report.html').write_text(page, encoding='utf-8')
    return df, sf
'''
(CODE_DIR / 'klein_report.py').write_text(module_source, encoding='utf-8')
print('Ready:', 'klein_report.py')

## 4. Built-in reference photo (no upload), check the actual prompts
The next cell holds the built-in photo as base64 (its code is collapsed by default in Colab). At run time it is decoded back to the original JPEG, its SHA-256 is verified,
and it is written to `images_klein_custom/baked/`. Only this image (plus images uploaded to `images_klein_custom/uploads/` when `ALSO_UPLOAD` is on) joins the experiment;
old images left in the folder are no longer mixed in.

Preprocessing is unchanged: EXIF rotation, RGB conversion and longest-side scaling (1600 → `REF_MAX_SIDE`); no face cropping or warping.
Every prompt you entered is applied to every reference image, and all precisions use exactly the same text and PNG.

To use a different built-in image: put the new image's base64 and sha256 in the next cell, or turn off `USE_BAKED_IMAGE` and upload instead.

In [ ]:
#@title Built-in reference photo (base64, embedded in the notebook; no upload needed)
# Original JPEG bytes, not re-encoded; the next cell verifies the sha256 when decoding.
BAKED_IMAGES = {
    'reference_photo.jpg': dict(
        sha256='971c11a13cffab9811229656adfc6224033cf1ac36a928e3f0ee8d35804bb157',
        width=1254, height=1254, nbytes=81736,
        base64=(
            '...'
        ),
    ),
}
for _name, _item in BAKED_IMAGES.items():
    print(f"Built-in reference {_name}: {_item['width']}×{_item['height']}, {_item['nbytes']/1024:.1f} KiB, "
          f"sha256 {_item['sha256'][:12]}…")

In [ ]:
from PIL import Image
import base64
import numpy as np
import pandas as pd
from IPython.display import display
import importlib.util

def load_module(name):
    spec = importlib.util.spec_from_file_location(name, CODE_DIR / (name + '.py'))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

kp = load_module('klein_prompts')
IMAGE_SUFFIXES = ('.jpg', '.jpeg', '.png', '.webp')
image_paths = []

if USE_BAKED_IMAGE:
    baked_dir = FACE_DIR / 'baked'
    baked_dir.mkdir(parents=True, exist_ok=True)
    for name, item in BAKED_IMAGES.items():
        data = base64.b64decode(item['base64'], validate=True)
        digest = hashlib.sha256(data).hexdigest()
        if digest != item['sha256'] or len(data) != item['nbytes']:
            raise RuntimeError(f'Built-in image {name} failed verification: the base64 in the notebook was modified or truncated.')
        path = baked_dir / name
        path.write_bytes(data)
        image_paths.append(path)
        print(f'Using built-in reference: {path.name} | {len(data)/1024:.1f} KiB | sha256 {digest[:12]}…')

if ALSO_UPLOAD:
    upload_dir = FACE_DIR / 'uploads'
    upload_dir.mkdir(parents=True, exist_ok=True)
    try:
        from google.colab import files
        for name, data in files.upload().items():
            (upload_dir / Path(name).name).write_bytes(data)
    except ImportError:
        print('Not running in Colab: put extra images in', upload_dir)
    uploaded = sorted(p for p in upload_dir.iterdir() if p.suffix.lower() in IMAGE_SUFFIXES)
    print(f'Extra uploaded images: {[p.name for p in uploaded]}')
    image_paths += uploaded

assert image_paths, 'No reference images: turn on USE_BAKED_IMAGE, or turn on ALSO_UPLOAD and upload images.'
names = [p.name for p in image_paths]
assert len(names) == len(set(names)), f'Duplicate reference file names; rename them and upload again: {names}'

cases = kp.make_cases(image_paths, OUT / 'inputs', PROMPTS, SEED, REF_MAX_SIDE)
for fi, path in enumerate(image_paths):
    first = cases[fi * len(PROMPTS)]
    print(path.name, '| reference:', first['reference_size'])
    with Image.open(first['reference_path']) as image:
        preview = image.copy()
    preview.thumbnail((256,256))
    display(preview)
print('These prompts go to the model verbatim, with nothing appended:')
for prompt_i, prompt in enumerate(PROMPTS, 1):
    print(f'\nPrompt {prompt_i}:\n{prompt}')
display(pd.DataFrame(cases)[['case_id','face','prompt_label','prompt']].style.set_properties(**{'white-space':'pre-wrap'}))
print(f'{len(cases)} cases × {len(PRECISIONS)} precisions × {REPEATS} repeats = {len(cases)*len(PRECISIONS)*REPEATS} timed images')

## 5. Download the original model once and pin its revision
Only the subfolders Diffusers needs are downloaded, avoiding the duplicate single-file checkpoint in the repo root.
If Hugging Face asks you to log in, run `from huggingface_hub import notebook_login; notebook_login()` separately before this cell.

In [ ]:
from huggingface_hub import HfApi, snapshot_download
if LOCAL_MODEL_PATH:
    model_path = str(Path(LOCAL_MODEL_PATH).expanduser().resolve())
    resolved_revision = 'local-directory'
else:
    resolved_revision = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION).sha
    model_path = snapshot_download(
        MODEL_ID, revision=resolved_revision,
        allow_patterns=['model_index.json','scheduler/*','tokenizer/*',
                        'text_encoder/*.json','text_encoder/*.safetensors',
                        'transformer/*.json','transformer/*.safetensors',
                        'vae/*.json','vae/*.safetensors'])
index = json.loads((Path(model_path) / 'model_index.json').read_text())
assert index['_class_name'] == 'Flux2KleinPipeline' and index.get('is_distilled') is True
assert index['text_encoder'][1] == 'Qwen3ForCausalLM'
config = dict(model_id=MODEL_ID, model_revision=resolved_revision, model_path=model_path,
              precisions=PRECISIONS, width=WIDTH, height=HEIGHT, reference_max_side=REF_MAX_SIDE,
              seed=SEED, warmup_runs=WARMUP_RUNS, repeats=REPEATS, steps=4, guidance_scale=1.0,
              compute_dtype='bfloat16', quantize_components=QUANTIZE_COMPONENTS, exclude=EXCLUDE,
              prompt_style=PROMPT_STYLE, cases=cases, output_dir=str(OUT), asset_dir=str(ASSET_DIR),
              source_notebook='klein_S2c_custom_prompts_baked_int4fast.ipynb', source_sha256='67c725e00852261d1f6b28911247908c16c983af27d9e4f484b9be0edef501ea',
              baked_images={n: dict(sha256=v['sha256'], width=v['width'], height=v['height'])
                            for n, v in BAKED_IMAGES.items()} if USE_BAKED_IMAGE else {},
              also_upload=ALSO_UPLOAD, int4_backend=INT4_BACKEND,
              experiment='custom_literal_prompts', grad_mode='no_grad')
CONFIG_PATH = OUT / 'config.json'
CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
(OUT / 'prompts.json').write_text(json.dumps(cases, ensure_ascii=False, indent=2), encoding='utf-8')
(OUT / 'requirements_freeze.txt').write_text(
    subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True), encoding='utf-8')
print('Model revision:', resolved_revision)
print('Config frozen at:', CONFIG_PATH)

## 6. Run the selected precisions one after another
Each group prints its matmul path, loading, quantization, warm-up and per-image results. The INT4 group starts with a numerical self-check and prints the single-layer time of the largest matmul (8064×3072×27648) in BF16 and in INT4, so you can confirm right away that INT4 is no longer slow.
**Timed:** tokenization and text encoding, VAE encoding of the reference, the 4 DiT steps, VAE decoding and the copy of the output to the CPU.
Not timed: download, loading, quantization, warm-up, PNG saving and identity evaluation.

Everything stays resident on the GPU by default; no CPU offload, torch.compile, VAE tiling or cached prompt embeddings.
If GPU memory runs out, the error goes into the log; resolution is never quietly lowered and offload is never quietly enabled for just one group.

All groups use `torch.no_grad()`. If you changed section 2, re-run from that section onward so the frozen config is updated.

In [ ]:
from IPython.display import clear_output
for precision in config['precisions']:
    mode_dir = OUT / precision
    # On a re-run, move old results to a timestamped backup so they are never mistaken for this run.
    if mode_dir.exists():
        backup = OUT.parent / f'{OUT.name}_previous_{precision}_{time.time_ns()}'
        mode_dir.rename(backup)
    mode_dir.mkdir(parents=True)
    print(f'\n=== {precision.upper()} ===', flush=True)
    env = os.environ.copy()
    env.update(PYTHONUNBUFFERED='1', TOKENIZERS_PARALLELISM='false', MAX_JOBS='2')
    cmd = [sys.executable, '-u', str(CODE_DIR / 'klein_quant_worker.py'),
           '--config', str(CONFIG_PATH), '--precision', precision]
    with (mode_dir / 'run.log').open('w', encoding='utf-8') as log:
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1, env=env)
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line); log.flush()
            rc = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill(); process.wait()
            raise
    status_path = mode_dir / 'status.json'
    if not status_path.exists():
        status_path.write_text(json.dumps(dict(status='failed', precision=precision,
                                              error=f'Worker exited with code {rc}; see run.log'),indent=2))
    print('Status:', json.loads(status_path.read_text())['status'])
print('All selected precisions finished. Even if a group failed, the report below still shows its error status.')

## 7. Optional: identity-preservation evaluation
Runs when `RUN_ID_METRICS=True`. Uses the original notebook's `w600k_r50.onnx`, MediaPipe five-point landmarks,
Umeyama alignment and ArcFace cosine similarity; the measurement runs separately on the CPU and is not counted in the generation time.
Failures to download the extra models or to detect faces do not block exporting the results. `id_sim` is not a probability of identity preservation, and it cannot be compared across inputs or body-shape changes out of context.
These InsightFace pretrained weights are for non-commercial research only; they are used here only for this optional test.

In [ ]:
if RUN_ID_METRICS:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                               'mediapipe==0.10.32', 'onnxruntime'])
        with (OUT / 'identity.log').open('w', encoding='utf-8') as log:
            ret = subprocess.run([sys.executable, str(CODE_DIR/'klein_identity.py'), str(OUT)],
                                 stdout=log, stderr=subprocess.STDOUT, text=True)
        print((OUT / 'identity.log').read_text(encoding='utf-8')[-3000:])
        if ret.returncode:
            print('Identity metrics did not finish; the images and the time/memory report are still available.')
    except Exception as exc:
        print('Identity evaluation skipped:', repr(exc))
else:
    print('Identity model not enabled; the cells below compare the reference and the BF16/INT8/INT4 images side by side.')

## 8. Summary table, image comparison and HTML report
- `kernel`: the path this group's matmuls actually take. INT8 and the default INT4 both "dequantize to BF16, then use cuBLAS", so they should be only slightly slower than BF16; `quanto_tinygemm` is 5–6x slower.
- `generation_median_s`: median generation time of the successful samples; per-image numbers are in `metrics.csv`.
- `paired_speedup_median`: BF16 time / this group's time, paired by input, prompt, seed and repeat index, then the median; below 1 means slower.
- `peak_allocated_gib`: peak GPU memory actually allocated by PyTorch, including the model and intermediate activations; not the same as the whole-process usage shown by `nvidia-smi`.
- `peak_reserved_gib`: GPU memory reserved by the PyTorch allocator; `resident_allocated_gib` is what stays resident after loading and quantizing the model.
- `tensor_storage_mib`: actual weight/buffer tensor storage, including packed integer data and quantization scales; not the original download size and not runtime memory.
- `load_s` / `quantize_s`: loading and quantization times, recorded separately and never mixed into the generation time.
- `mae_vs_bf16` / `psnr_vs_bf16_db`: pixel drift relative to BF16, not a measure of generation quality or identity.

**Quantization first of all reduces weight storage; it is not necessarily faster on an A100.** BF16 Tensor Cores, unpacking overhead and kernel support all affect speed.
This test cannot predict the INT8/INT4 speedup on a phone NPU. The three groups use the same random seed, but different precisions are not guaranteed to be pixel-identical.

In [ ]:
report_module = load_module('klein_report')
metrics, summary = report_module.build_report(OUT)
display(summary.round(3))
for case in config['cases']:
    print(case['face'], '|', case['prompt_label'])
    print(case['prompt'])
    display(Image.open(OUT / f'grid_{case["case_id"]}.png'))
print('Report with full prompts and full-size images:', OUT / 'report.html')
print('Per-image data:', OUT / 'metrics.csv')
if summary['n_ok'].sum() == 0:
    print('Note: no image was generated successfully. Check run.log/status.json in each precision folder; the ZIP can still be downloaded for debugging.')

## 9. Package and download
The ZIP contains the reference images, the outputs of all groups, comparison grids, HTML, CSV, config, model revision, library versions, per-layer quantization records and error logs.
The model checkpoint is not included, so you will not re-download over ten GB. Unzip the whole archive before opening the HTML.

In [ ]:
zip_path = shutil.make_archive(str(OUT), 'zip', root_dir=OUT)
print('ZIP:', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    from IPython.display import FileLink
    display(FileLink(zip_path))


## Sources and scope
- [FLUX.2 klein 4B model card](https://huggingface.co/black-forest-labs/FLUX.2-klein-4B)
- [Flux2KleinPipeline source](https://github.com/huggingface/diffusers/blob/main/src/diffusers/pipelines/flux2/pipeline_flux2_klein.py)
- [Quanto quantize / freeze / quantization_map](https://github.com/huggingface/optimum-quanto)
- [Reported Quanto INT4 incompatibility with inference_mode](https://github.com/huggingface/optimum-quanto/issues/304)

The reference photo is embedded as base64 and its SHA-256 is verified at run time. On top of that, this version only changes the INT4 matmul path (new `klein_w4a16.py`); the BF16 and INT8 code is unchanged:
- Quantization: calls Quanto 0.2.7's own `MaxOptimizer` and `quantize_affine`, one group per 128 input channels; int4 values, scales and shifts are bit-identical to Quanto's.
- Storage: two int4 values per byte, plus a bf16 scale and zero per group: 0.53 bytes per parameter, the same size as Quanto's tinygemm format.
- Compute: each layer's forward pass first expands the weights to BF16 with three elementwise kernels, then calls `F.linear` (cuBLAS). Dequantization reproduces tinygemm's GPU arithmetic `bf16(fma(q-8, scale, zero))`; layers with `in_features ≤ 128` (the DiT's `x_embedder`) reproduce Quanto's generic formula. In CPU validation it matched tinygemm's single-rounding arithmetic bit for bit, so compared with the previous INT4 version only the matmul kernel changed, and the output images should be essentially unchanged (only tiny differences from the matmul accumulation order).
- No Quanto CUDA extension needs to be compiled.

This version removes the body-shape keyword parser, the gender setting and the automatic prompt template, and keeps the user's input verbatim.
It is still image editing with the original photo as the reference, not video generation; how well the output follows the prompt has to be tested.
The Qwen3 4B text encoder is still used; no small-model feature-alignment experiment is included.

Quanto is pinned to `0.2.7`, using integer weight quantization with BF16 activations; all three groups use `no_grad`.
The INT4 compatibility changes need to be re-tested on your A100; compatibility with every runtime version is not guaranteed.
Failures are saved and the report can still be exported; precision, resolution and quantization backend are never changed automatically.
Older versions used a different prompt template and `inference_mode`; compare precisions within the same version, and do not attribute the whole timing difference between versions to quantization.